# 🚀 RizeHire — BERT Fine-Tuning for Resume-Job Matching

**Project:** RizeHire (AI-Powered Recruitment Platform)
**Student:** Sandeep Kumar (2022UG1113, IIIT Ranchi)
**Supervisor:** Dr. Roshan Kr. Singh

This notebook fine-tunes a BERT model on resume-job matching data using a free Colab GPU.
Instead of using frozen S-BERT embeddings (which gave ~58% accuracy), we fine-tune
the actual BERT transformer weights to learn domain-specific resume matching patterns.

### How to use:
1. **Runtime → Change runtime type → GPU (T4)**
2. Upload `dataset.csv` when prompted
3. Click **Runtime → Run all**
4. Download the 3 output files at the end to your `python-explainability/` folder

## Step 1: Install Dependencies & Check GPU

In [ ]:
!pip install -q sentence-transformers torch pandas scikit-learn

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU (T4)")

## Step 2: Upload Dataset

Run this cell — it will show an upload button. Click it and select your `dataset.csv` file.

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()  # Upload button will appear — select dataset.csv

# Load the dataset
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f"✅ Loaded {filename}: {len(df)} rows")
print(f"   Columns: {list(df.columns)}")
print(f"   Decision distribution: {dict(df['decision'].value_counts())}")

## Step 3: Prepare Data for Fine-Tuning

We combine Resume + Interview Transcript as the candidate text, paired with the Job Description.
The BERT cross-encoder will process both texts **together** through the transformer — unlike the frozen bi-encoder approach which encoded them separately.

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Map labels
label_map = {
    'select': 1, 'hire': 1, 'hired': 1, 'yes': 1, 'accept': 1,
    'reject': 0, 'rejected': 0, 'no': 0, 'decline': 0
}
df['label'] = df['decision'].astype(str).str.lower().str.strip().map(label_map)
df = df.dropna(subset=['label', 'Resume', 'Job_Description'])
df['label'] = df['label'].astype(int)

# Build candidate text: Resume + Transcript (truncated to fit BERT's 512 token window)
# We give resume ~300 tokens worth of chars and transcript ~200 tokens worth
if 'Transcript' in df.columns:
    df['candidate_text'] = (
        df['Resume'].astype(str).str[:1500] +
        " [SEP] Interview highlights: " +
        df['Transcript'].astype(str).str[:1000]
    )
    print("✅ Combined Resume + Transcript as candidate text")
else:
    df['candidate_text'] = df['Resume'].astype(str).str[:2500]
    print("✅ Using Resume only as candidate text")

df['jd_text'] = df['Job_Description'].astype(str).str[:500]

# Train/test split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"\n📊 Dataset ready:")
print(f"   Train: {len(train_df)} samples")
print(f"   Validation: {len(val_df)} samples")
print(f"   Label balance (train): {dict(train_df['label'].value_counts())}")
print(f"   Avg candidate text length: {df['candidate_text'].str.len().mean():.0f} chars")
print(f"   Avg JD text length: {df['jd_text'].str.len().mean():.0f} chars")

## Step 4: Fine-Tune BERT Cross-Encoder

This is where the magic happens. Instead of freezing BERT and only training a small classifier,
we **unfreeze BERT's transformer layers** and train them on our resume-job matching task.

The cross-encoder processes `[CLS] candidate_text [SEP] job_description [SEP]` as a single input,
allowing BERT to attend across both texts simultaneously.

**Expected time:** ~10-20 minutes on T4 GPU

In [ ]:
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
import math

# ─── Configuration ───
MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
EPOCHS = 4
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

print(f"📥 Loading cross-encoder: {MODEL_NAME}")
print(f"   This model will be FINE-TUNED on your resume-job matching data")
print()

# ─── Prepare training data as InputExamples ───
train_examples = []
for _, row in train_df.iterrows():
    train_examples.append(InputExample(
        texts=[row['candidate_text'], row['jd_text']],
        label=float(row['label'])
    ))

# Validation pairs for evaluator
val_sentences1 = val_df['candidate_text'].tolist()
val_sentences2 = val_df['jd_text'].tolist()
val_labels = val_df['label'].tolist()

# Create DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
total_steps = math.ceil(len(train_examples) / BATCH_SIZE) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

# Create evaluator (handle both old and new API versions)
try:
    from sentence_transformers.cross_encoder.evaluation import CrossEncoderClassificationEvaluator
    import pandas as pd_eval
    val_eval_df = pd.DataFrame({
        "sentence1": val_sentences1,
        "sentence2": val_sentences2,
        "label": val_labels
    })
    evaluator = CrossEncoderClassificationEvaluator(
        samples=[val_eval_df],
        name='rizehire-val'
    )
    print("✅ Using new CrossEncoderClassificationEvaluator API")
except (ImportError, TypeError):
    try:
        from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
        evaluator = CEBinaryClassificationEvaluator(
            val_sentences1, val_sentences2, val_labels,
            name='rizehire-val'
        )
        print("✅ Using legacy CEBinaryClassificationEvaluator API")
    except TypeError:
        evaluator = None
        print("⚠️ Evaluator not available, training without validation evaluation")

print(f"\n✅ Data prepared:")
print(f"   Train examples: {len(train_examples)}")
print(f"   Total training steps: {total_steps}")
print(f"   Warmup steps: {warmup_steps}")
print()

# ─── Initialize Cross-Encoder ───
model = CrossEncoder(MODEL_NAME, num_labels=1, max_length=512)
print(f"✅ Cross-encoder loaded")
print(f"   All BERT layers will be fine-tuned (not frozen!)")
print()

# ─── Train! ───
print("🔄 Starting fine-tuning...")
print("=" * 60)

fit_kwargs = dict(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LEARNING_RATE},
    output_path='./rizehire_cross_encoder',
    save_best_model=True,
    show_progress_bar=True,
)
if evaluator is not None:
    fit_kwargs['evaluator'] = evaluator

model.fit(**fit_kwargs)

print()
print("=" * 60)
print("✅ Fine-tuning complete!")

## Step 5: Evaluate the Fine-Tuned Model

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import os

# Use the model we just trained (already in memory) — no need to reload from disk
best_model = model

# Get predictions on validation set
val_pairs = list(zip(val_df['candidate_text'].tolist(), val_df['jd_text'].tolist()))
val_scores = best_model.predict(val_pairs, show_progress_bar=True)

# Convert scores to binary predictions (threshold = 0.5)
val_preds = (np.array(val_scores) > 0.5).astype(int)
val_true = val_df['label'].values

# Results
final_acc = accuracy_score(val_true, val_preds)
final_f1 = f1_score(val_true, val_preds)

print("=" * 60)
print("📈 FINE-TUNED BERT MODEL — FINAL RESULTS")
print("=" * 60)
print()
print(classification_report(val_true, val_preds, target_names=['Reject', 'Select']))
print(f"   ✅ Accuracy: {final_acc:.1%}")
print(f"   ✅ F1 Score: {final_f1:.4f}")
print()

# Confusion matrix
cm = confusion_matrix(val_true, val_preds)
print(f"   Confusion Matrix:")
print(f"                  Predicted Reject  Predicted Select")
print(f"   Actual Reject:       {cm[0][0]:>5}          {cm[0][1]:>5}")
print(f"   Actual Select:       {cm[1][0]:>5}          {cm[1][1]:>5}")

# Compare with frozen approach
print()
print("=" * 60)
print(f"   📊 Frozen S-BERT (before):  57.8% accuracy")
print(f"   📊 Fine-tuned BERT (after): {final_acc:.1%} accuracy")
print(f"   📈 Improvement: +{(final_acc - 0.578)*100:.1f} percentage points")
print("=" * 60)

## Step 6: Export Model for RizeHire Flask Service

This generates the 3 files your `app.py` Flask service needs:
1. `bert_matching_model.pt` — the fine-tuned model weights
2. `background_data.pkl` — background samples for SHAP/LIME
3. `bert_model_metadata.json` — training metadata

After download, place all 3 files in your `python-explainability/` folder.

In [ ]:
import pickle
import json
import os
from datetime import datetime
from sentence_transformers import SentenceTransformer

# ─── Generate S-BERT embeddings for background data (needed by SHAP/LIME in app.py) ───
print("📥 Loading S-BERT for generating background embeddings...")
sbert = SentenceTransformer('all-MiniLM-L6-v2')

# Take 100 random training samples for SHAP background
bg_sample = train_df.sample(n=min(100, len(train_df)), random_state=42)
print("🧠 Generating background embeddings...")
bg_resume_emb = sbert.encode(bg_sample['candidate_text'].tolist(), normalize_embeddings=True, show_progress_bar=False)
bg_job_emb = sbert.encode(bg_sample['jd_text'].tolist(), normalize_embeddings=True, show_progress_bar=False)

# ─── Save 1: bert_matching_model.pt ───
ce_model = best_model.model  # The HuggingFace model inside the CrossEncoder

model_save = {
    'model_state_dict': ce_model.state_dict(),
    'config': {
        'embedding_dim': 384,
        'hidden_dim': 256,
        'dropout': 0.4,
        'sbert_model': 'all-MiniLM-L6-v2',
        'cross_encoder_model': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
        'model_type': 'cross-encoder-finetuned',
    },
    'metrics': {
        'accuracy': float(final_acc),
        'f1_score': float(final_f1),
        'train_samples': len(train_df),
        'val_samples': len(val_df),
    },
}
torch.save(model_save, 'bert_matching_model.pt')
print("💾 Saved: bert_matching_model.pt")

# Save the full cross-encoder directory using save_pretrained
save_dir = '/content/rizehire_cross_encoder_final'
os.makedirs(save_dir, exist_ok=True)
ce_model.save_pretrained(save_dir)
# Also save tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2')
tokenizer.save_pretrained(save_dir)
print("💾 Saved: rizehire_cross_encoder_final/ (full model directory)")

# ─── Save 2: background_data.pkl ───
background_data = {
    'resume_embeddings': bg_resume_emb.tolist(),
    'job_embeddings': bg_job_emb.tolist(),
    'labels': bg_sample['label'].tolist(),
}
with open('background_data.pkl', 'wb') as f:
    pickle.dump(background_data, f)
print("💾 Saved: background_data.pkl")

# ─── Save 3: bert_model_metadata.json ───
metadata = {
    'trained_at': datetime.now().isoformat(),
    'dataset': 'dataset.csv',
    'total_samples': len(df),
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'accuracy': float(final_acc),
    'f1_score': float(final_f1),
    'sbert_model': 'all-MiniLM-L6-v2',
    'cross_encoder_model': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
    'embedding_dim': 384,
    'architecture': 'Cross-Encoder (MiniLM-L6 fine-tuned)',
    'model_type': 'cross-encoder-finetuned',
    'training': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'warmup_ratio': WARMUP_RATIO,
        'platform': 'Google Colab (T4 GPU)',
    },
    'total_parameters': sum(p.numel() for p in ce_model.parameters()),
}
with open('bert_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("💾 Saved: bert_model_metadata.json")

print()
print("=" * 60)
print("✅ All files ready for download!")
print("=" * 60)

## Step 7: Download Files

Run this cell to download all 3 files. Then copy them to your `python-explainability/` folder.

Also downloads the full cross-encoder model directory as a zip (needed for `app.py` to load directly).

In [ ]:
import shutil
from google.colab import files

# Zip the cross-encoder model directory
shutil.make_archive('rizehire_cross_encoder_final', 'zip', '/content', 'rizehire_cross_encoder_final')

# Download all files
print("📥 Starting downloads...")
print("   (Your browser will download 4 files)")
print()

files.download('bert_matching_model.pt')
files.download('background_data.pkl')
files.download('bert_model_metadata.json')
files.download('rizehire_cross_encoder_final.zip')

print()
print("=" * 60)
print("🎉 ALL DONE!")
print("=" * 60)
print()
print("📋 Next steps:")
print("   1. Unzip 'rizehire_cross_encoder_final.zip' into your")
print("      python-explainability/ folder")
print("   2. Copy the other 3 files there too")
print("   3. Your folder should look like:")
print("      python-explainability/")
print("      ├── app.py")
print("      ├── train_bert_model.py")
print("      ├── bert_matching_model.pt")
print("      ├── background_data.pkl")
print("      ├── bert_model_metadata.json")
print("      ├── rizehire_cross_encoder_final/")
print("      │   ├── config.json")
print("      │   ├── model.safetensors")
print("      │   ├── tokenizer_config.json")
print("      │   └── ...")
print("      └── requirements.txt")
print()
print("   4. Run: python app.py")
print("   5. The Flask service will auto-load the fine-tuned model!")